In [2]:
# Step 0: Install Ultralytics YOLO and basic deps, then confirm GPU works
!pip -q install ultralytics kagglehub opencv-python

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.3 MB/s eta 0:00:00
Torch: 2.9.0+cu128
CUDA available: True


In [3]:
# Step 1: Download the Kaggle dataset using kagglehub (fastest for Colab)
import kagglehub

# This pulls: orvile/x-ray-baggage-anomaly-detection
dataset_path = kagglehub.dataset_download("orvile/x-ray-baggage-anomaly-detection")
print("Dataset downloaded to:", dataset_path)


Using Colab cache for faster access to the 'x-ray-baggage-anomaly-detection' dataset.
Dataset downloaded to: /kaggle/input/x-ray-baggage-anomaly-detection


In [4]:
# Step 2: Locate the dataset's data.yaml and print it for sanity checking
import os, glob

yamls = glob.glob(os.path.join(dataset_path, "**", "data.yaml"), recursive=True)
print("Found YAML files:", yamls)

assert len(yamls) > 0, "No data.yaml found. Dataset structure may be different than expected."
data_yaml = yamls[0]
print("Using:", data_yaml)

with open(data_yaml, "r") as f:
    print(f.read())


Found YAML files: ['/kaggle/input/x-ray-baggage-anomaly-detection/data.yaml']
Using: /kaggle/input/x-ray-baggage-anomaly-detection/data.yaml
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 5
names: ['0', '1', '2', '3', '4']

roboflow:
  workspace: malek-mhnrl
  project: x-ray-baggage-detection
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/malek-mhnrl/x-ray-baggage-detection/dataset/1


In [5]:
# Step 3: Create a Colab-safe data.yaml with absolute paths (only if needed)
import yaml

with open(data_yaml, "r") as f:
    cfg = yaml.safe_load(f)

# Make sure 'path' points to the actual dataset_path
cfg["path"] = dataset_path

# Write a new yaml we control
patched_yaml = "/content/xray_data.yaml"
with open(patched_yaml, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Wrote patched YAML to:", patched_yaml)
print("\n--- patched YAML ---")
with open(patched_yaml, "r") as f:
    print(f.read())


Wrote patched YAML to: /content/xray_data.yaml

--- patched YAML ---
train: ../train/images
val: ../valid/images
test: ../test/images
nc: 5
names:
- '0'
- '1'
- '2'
- '3'
- '4'
roboflow:
  workspace: malek-mhnrl
  project: x-ray-baggage-detection
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/malek-mhnrl/x-ray-baggage-detection/dataset/1
path: /kaggle/input/x-ray-baggage-anomaly-detection



In [ ]:
# Step 4: Train a lightweight YOLO model (YOLOv8n) for real-time use
from ultralytics import YOLO

CONFIG = patched_yaml  # change to data_yaml if you skipped patching

EPOCHS = 25
IMG_SIZE = 512      # lower = faster; 512 is a good balance for T4
BATCH = 16          # if OOM, try 8
DEVICE = 0 if torch.cuda.is_available() else "cpu"

model = YOLO("yolov8n.pt")

results = model.train(
    data=CONFIG,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    name="xray_baggage_yolov8n",
    patience=10,
    cache=False,
    # X-ray-friendly augmentation choices
    flipud=0.0,      # vertical flip often unrealistic for baggage scans
    fliplr=0.5,
    degrees=10,
)

print("Saved run to:", results.save_dir)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/xray_data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, i

In [1]:
# Step 5: Evaluate the model on validation set (mAP, precision/recall etc.)
val = model.val(data=CONFIG, imgsz=IMG_SIZE, device=DEVICE)
print("mAP50:", float(val.box.map50))
print("mAP50-95:", float(val.box.map))


NameError: name 'model' is not defined

In [7]:
# Step 6: Run inference on one or more image URLs and return structured JSON
import requests
import numpy as np
import cv2
from typing import List, Dict, Any

def detect_from_url(model, url: str, imgsz: int = 512, conf: float = 0.25) -> Dict[str, Any]:
    # Download image
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    data = np.frombuffer(r.content, dtype=np.uint8)

    # Decode image (handles jpg/png)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        return {"url": url, "error": "Could not decode image"}

    # Run YOLO
    pred = model.predict(img, imgsz=imgsz, conf=conf, verbose=False)[0]

    # Convert results to JSON-friendly list
    dets = []
    if pred.boxes is not None and len(pred.boxes) > 0:
        boxes = pred.boxes.xyxy.cpu().numpy()
        confs = pred.boxes.conf.cpu().numpy()
        clss  = pred.boxes.cls.cpu().numpy().astype(int)
        names = pred.names

        for (x1,y1,x2,y2), c, k in zip(boxes, confs, clss):
            dets.append({
                "class_id": int(k),
                "class_name": names[int(k)],
                "confidence": float(c),
                "bbox_xyxy": [float(x1), float(y1), float(x2), float(y2)],
            })

    return {"url": url, "detections": dets}

def detect_many_urls(model, urls: List[str], imgsz: int = 512, conf: float = 0.25):
    return [detect_from_url(model, u, imgsz=imgsz, conf=conf) for u in urls]

# Example usage:
# urls = ["https://.../some_xray.png", "https://.../another.png"]
# outputs = detect_many_urls(model, urls)
# outputs


In [8]:
# Step 7: Export to ONNX for faster inference in production (optional)
model.export(format="onnx", opset=12)
print("Exported ONNX.")


Ultralytics 8.4.13 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/content/runs/detect/xray_baggage_yolov8n/weights/best.pt' with input shape (1, 3, 512, 512) BCHW and output shape(s) (1, 9, 5376) (5.9 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 193ms
Prepared 4 packages in 3.52s
Installed 4 packages in 249ms
 + colorama==0.4.6
 + onnx==1.20.1
 + onnxruntime-gpu==1.24.1
 + onnxslim==0.1.84

requirements: AutoUpdate success ✅ 4.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 12...
ONNX: slimming with onnxslim 0.1.84...
ONNX: export success ✅ 5.9s, saved as '